# 01 — LiDAR & Point Cloud 기초

> **참고:** Udacity SFND_Lidar_Obstacle_Detection
> **언어:** C++ (PCL), Python (Open3D)
> **데이터:** KITTI `.bin` 포인트 클라우드

---

## 1-1. LiDAR란?

LiDAR(Light Detection And Ranging)는 레이저 펄스를 발사해 반사 시간(ToF)으로 거리를 측정한다.



```
거리 d = (c × Δt) / 2
  c  : 빛의 속도 (3×10⁸ m/s)
  Δt : 왕복 비행 시간
```




**Velodyne HDL-64E** (KITTI 기준):
- 64개 레이저 채널, 수직 FoV ±26.8°
- 360° 수평 회전, 10 Hz
- 출력: `(x, y, z, intensity)` 포인트 배열

---

## 1-2. 포인트 클라우드 로딩

### Python (Open3D)



In [ ]:
import numpy as np
import open3d as o3d

def load_kitti_bin(path: str) -> np.ndarray:
    """KITTI .bin → (N, 4) float32: x, y, z, intensity"""
    pts = np.fromfile(path, dtype=np.float32).reshape(-1, 4)
    return pts

def to_o3d(pts: np.ndarray) -> o3d.geometry.PointCloud:
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts[:, :3])
    return pcd

pts = load_kitti_bin("0000000000.bin")
print(f"Points: {len(pts)}")   # ~120,000
pcd = to_o3d(pts)
o3d.visualization.draw_geometries([pcd])




### C++ (PCL)



```cpp
#include <pcl/io/pcd_io.h>
#include <pcl/point_types.h>

pcl::PointCloud<pcl::PointXYZI>::Ptr cloud(new pcl::PointCloud<pcl::PointXYZI>);
pcl::io::loadPCDFile("cloud.pcd", *cloud);
std::cout << "Loaded " << cloud->size() << " points\n";
```




---

## 1-3. Voxel Grid 다운샘플링

포인트 클라우드는 수십만 개 → 처리 전 다운샘플링 필수.



In [ ]:
# Python (Open3D)
pcd_down = pcd.voxel_down_sample(voxel_size=0.2)  # 20 cm 격자
print(f"Downsampled: {len(pcd_down.points)}")


```cpp
// C++ (PCL)
#include <pcl/filters/voxel_grid.h>

pcl::VoxelGrid<pcl::PointXYZI> vg;
vg.setInputCloud(cloud);
vg.setLeafSize(0.2f, 0.2f, 0.2f);
pcl::PointCloud<pcl::PointXYZI>::Ptr cloud_filtered(new pcl::PointCloud<pcl::PointXYZI>);
vg.filter(*cloud_filtered);
```




**원리:** 3D 공간을 voxel로 분할 → 각 voxel 내 포인트를 centroid 하나로 대체

---

## 1-4. RANSAC 지면 분리

자율주행에서 지면(plane)은 관심 없음 → 먼저 분리.

### 알고리즘



```
1. 랜덤하게 3점 선택
2. 해당 3점으로 평면 방정식 Ax+By+Cz+D=0 결정
3. 나머지 포인트 중 |Ax+By+Cz+D|/√(A²+B²+C²) < threshold 인 것 = inliers
4. inliers 수가 최대인 평면 선택
5. 반복 횟수: n = log(1-p) / log(1-(1-e)^s)
   p=0.99, e=outlier비율, s=3(sample 수)
```


In [ ]:
import numpy as np

def fit_plane_ransac(pts: np.ndarray, n_iter=100, dist_thresh=0.2):
    best_inliers = np.array([], dtype=int)
    n = len(pts)
    for _ in range(n_iter):
        idx = np.random.choice(n, 3, replace=False)
        p1, p2, p3 = pts[idx, :3]
        # 평면 법선 벡터
        v1, v2 = p2 - p1, p3 - p1
        normal = np.cross(v1, v2)
        A, B, C = normal
        D = -(normal @ p1)
        denom = np.sqrt(A**2 + B**2 + C**2) + 1e-9
        dist = np.abs(pts[:, :3] @ np.array([A, B, C]) + D) / denom
        inliers = np.where(dist < dist_thresh)[0]
        if len(inliers) > len(best_inliers):
            best_inliers = inliers
    ground = pts[best_inliers]
    obstacles = pts[np.setdiff1d(np.arange(n), best_inliers)]
    return ground, obstacles

ground, obstacles = fit_plane_ransac(pts, n_iter=100, dist_thresh=0.2)
print(f"Ground: {len(ground)}, Obstacles: {len(obstacles)}")




---

## 1-5. Euclidean Clustering (장애물 군집화)

지면 제거 후 남은 포인트를 개별 물체로 분리.

### KD-Tree 기반 거리 탐색



In [ ]:
from scipy.spatial import KDTree

def euclidean_cluster(pts: np.ndarray, eps=0.5, min_pts=10):
    """
    eps      : 이웃 탐색 반경 (m)
    min_pts  : 클러스터 최소 포인트 수
    """
    tree = KDTree(pts[:, :3])
    visited = np.zeros(len(pts), dtype=bool)
    clusters = []

    for i in range(len(pts)):
        if visited[i]:
            continue
        neighbors = tree.query_ball_point(pts[i, :3], eps)
        if len(neighbors) < min_pts:
            visited[i] = True
            continue
        cluster = set()
        queue = list(neighbors)
        while queue:
            j = queue.pop()
            if visited[j]:
                continue
            visited[j] = True
            cluster.add(j)
            new_neighbors = tree.query_ball_point(pts[j, :3], eps)
            if len(new_neighbors) >= min_pts:
                queue.extend(new_neighbors)
        clusters.append(list(cluster))
    return clusters

clusters = euclidean_cluster(obstacles, eps=0.5, min_pts=10)
print(f"Found {len(clusters)} clusters")




---

## 1-6. 3D Bounding Box



In [ ]:
def bounding_box(pts: np.ndarray):
    """축 정렬 바운딩 박스 (AABB)"""
    min_pt = pts[:, :3].min(axis=0)
    max_pt = pts[:, :3].max(axis=0)
    center = (min_pt + max_pt) / 2
    size   = max_pt - min_pt
    return center, size

for i, cluster_idx in enumerate(clusters):
    cluster_pts = obstacles[cluster_idx]
    center, size = bounding_box(cluster_pts)
    print(f"Cluster {i}: center={center.round(2)}, size={size.round(2)}")




---

## 1-7. 전체 파이프라인 (Python)



In [ ]:
# pipeline.py
import numpy as np
import open3d as o3d

pts = load_kitti_bin("0000000000.bin")

# 1. ROI 필터 (자차 근처만)
roi_mask = (
    (pts[:, 0] > -20) & (pts[:, 0] < 50) &
    (pts[:, 1] > -10) & (pts[:, 1] < 10) &
    (pts[:, 2] > -3)  & (pts[:, 2] < 2)
)
pts = pts[roi_mask]

# 2. Voxel 다운샘플 (Open3D)
pcd = to_o3d(pts)
pcd = pcd.voxel_down_sample(0.2)
pts = np.asarray(pcd.points)

# 3. RANSAC 지면 분리
ground, obstacles = fit_plane_ransac(pts, n_iter=100, dist_thresh=0.2)

# 4. Euclidean Clustering
clusters = euclidean_cluster(obstacles, eps=0.5, min_pts=10)

# 5. BBox
boxes = [bounding_box(obstacles[c]) for c in clusters if len(c) > 5]
print(f"Detected {len(boxes)} objects")




---

## 1-8. C++ PCL 파이프라인 구조 (Udacity 스타일)



```cpp
// processPointClouds.cpp 핵심 구조
template<typename PointT>
std::pair<typename pcl::PointCloud<PointT>::Ptr,
          typename pcl::PointCloud<PointT>::Ptr>
ProcessPointClouds<PointT>::SegmentPlane(
    typename pcl::PointCloud<PointT>::Ptr cloud,
    int maxIterations, float distanceThreshold)
{
    pcl::SACSegmentation<PointT> seg;
    seg.setOptimizeCoefficients(true);
    seg.setModelType(pcl::SACMODEL_PLANE);
    seg.setMethodType(pcl::SAC_RANSAC);
    seg.setMaxIterations(maxIterations);
    seg.setDistanceThreshold(distanceThreshold);

    pcl::PointIndices::Ptr inliers(new pcl::PointIndices);
    pcl::ModelCoefficients::Ptr coefficients(new pcl::ModelCoefficients);
    seg.setInputCloud(cloud);
    seg.segment(*inliers, *coefficients);

    // Extract plane & obstacles
    pcl::ExtractIndices<PointT> extract;
    extract.setInputCloud(cloud);
    extract.setIndices(inliers);

    typename pcl::PointCloud<PointT>::Ptr planeCloud(new pcl::PointCloud<PointT>);
    extract.setNegative(false);
    extract.filter(*planeCloud);

    typename pcl::PointCloud<PointT>::Ptr obstCloud(new pcl::PointCloud<PointT>);
    extract.setNegative(true);
    extract.filter(*obstCloud);

    return {obstCloud, planeCloud};
}
```




---

## 핵심 개념 정리

| 개념 | 핵심 |
|------|------|
| Voxel Grid | 3D 격자 → centroid 대표, O(N) 복잡도 |
| RANSAC | 이상치에 강건한 평면/직선 피팅, 반복 횟수로 신뢰도 조절 |
| KD-Tree | k-nearest neighbor 쿼리 O(log N), 클러스터링 핵심 |
| Euclidean Clustering | DBSCAN 단순화 버전 (eps 고정, min_pts 조건) |
| AABB | 축 정렬 바운딩 박스, 빠르지만 회전 무시 |

---

## 다음 단계

→ **02_camera_basics.md** : Harris, SIFT, ORB 피처 검출 및 매칭
